# Interactive Map with Folium

Markers and circles for each launch site, a MarkerCluster of launch outcomes (green = success, red = failure), mouse-position readout and distance lines to nearby infrastructure.

> Proximity coordinates are **approximate** points picked from the map; the distance printed below is computed with the haversine formula.

In [1]:
import folium, pandas as pd, numpy as np
from folium.plugins import MarkerCluster, MousePosition
from math import radians, sin, cos, asin, sqrt
sites=pd.DataFrame({"Launch Site":["CCAFS LC-40","CCAFS SLC-40","KSC LC-39A","VAFB SLC-4E"],"Lat":[28.562302,28.563197,28.573255,34.632834],"Long":[-80.577356,-80.576820,-80.646895,-120.610746]})
sites

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610746


In [2]:
m=folium.Map(location=[28.56,-80.58],zoom_start=5)
for _,r in sites.iterrows():
    folium.Circle([r.Lat,r.Long],radius=1000,color="#0B49CB",fill=True).add_child(folium.Popup(r["Launch Site"])).add_to(m)
    folium.map.Marker([r.Lat,r.Long],icon=folium.features.DivIcon(icon_size=(20,20),icon_anchor=(0,0),html=f'<div style="font-size:12px;color:#0B49CB"><b>{r["Launch Site"]}</b></div>')).add_to(m)
m.save("figures/folium_sites.html"); m

In [3]:
# Launch outcomes per site (from the dashboard dataset)
dash=pd.read_csv("data/spacex_launch_dash.csv").merge(sites,on="Launch Site")
cluster=MarkerCluster().add_to(folium.Map(location=[28.56,-80.58],zoom_start=9))
m2=folium.Map(location=[28.56,-80.58],zoom_start=9); cluster=MarkerCluster().add_to(m2)
for _,r in dash.iterrows():
    folium.Marker([r.Lat,r.Long],icon=folium.Icon(color="white",icon_color="green" if r["class"]==1 else "red")).add_to(cluster)
MousePosition().add_to(m2)
m2.save("figures/folium_outcomes.html")
dash.groupby("Launch Site")["class"].agg(["sum","count","mean"])

,sum,count,mean
Launch Site,,,
CCAFS LC-40,7,26,0.269231
CCAFS SLC-40,3,7,0.428571
KSC LC-39A,10,13,0.769231
VAFB SLC-4E,4,10,0.400000


In [4]:
def haversine(lat1,lon1,lat2,lon2):
    lat1,lon1,lat2,lon2=map(radians,[lat1,lon1,lat2,lon2]); a=sin((lat2-lat1)/2)**2+cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return 6371*2*asin(sqrt(a))
site=(28.563197,-80.576820)
points={"Coastline":(28.56367,-80.56794),"Railway":(28.57205,-80.58527),"Highway":(28.56321,-80.57079),"City (Melbourne)":(28.1036,-80.6470)}
m3=folium.Map(location=site,zoom_start=13)
folium.Marker(site,popup="CCAFS SLC-40").add_to(m3)
for k,(la,lo) in points.items():
    dist=haversine(*site,la,lo); print(f"{k}: {dist:.2f} km")
    folium.PolyLine([site,(la,lo)],weight=2).add_to(m3)
    folium.Marker((la,lo),icon=folium.DivIcon(html=f'<div style="font-size:12px;color:#0A2A66"><b>{dist:.2f} km</b></div>')).add_to(m3)
m3.save("figures/folium_proximity.html")

Coastline: 0.87 km
Railway: 1.28 km
Highway: 0.59 km
City (Melbourne): 51.56 km
